# Task 04 — 데이터 주도 페르소나 정교화 (K-means)

> **목적**: Task 03의 규칙기반 2축 4사분면 페르소나를, 데이터 주도 K-means 군집으로 **교차검증·정교화**한다.
> SSOT = `src/news_health_features.py` (Task 02·03 확정 함수 재사용).

## 설계 결정 (사용자 확정)
| 갈림길 | 결정 | 근거(실데이터) |
|---|---|---|
| **군집 수 k** | **k=4** | 실루엣 0.35대 평탄, 4사분면과 1:1 정합·해석가능 |
| **군집 입력** | **2지표(신뢰·다양성)만** | 검증·회피 추가 시 실루엣 0.35→0.25 하락 + 표본 30% 손실 |
| **세그먼트 경계** | **하이브리드** | 운영·웹데모=중앙값 4사분면 규칙(재현·설명 명확), K-means=데이터 독립 재발견 **타당화 근거** |
| 표준화 | z (StandardScaler) | 다양성 우편향 → 표준화 필수 |

**핵심 결과 미리보기**: K-means(k=4)가 규칙기반 4사분면을 **ARI 0.56·라벨 일치 79.7%로 독립 재발견** →
"중앙값 임계는 임의가 아니라 데이터 군집 경계와 일치한다"는 Before/After 서사.


In [1]:
# ── 0. 셋업 ────────────────────────────────────────────────────────
import sys, numpy as np, pandas as pd
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src/news_health_features.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import news_health_features as F

pd.set_option("display.width", 120); pd.set_option("display.max_columns", 20)
df = F.load_2025()
w = df[F.WEIGHT]
print(f"N = {len(df):,} | 변수 {df.shape[1]} | WT 평균 {w.mean():.3f} (모집단 가중)")


N = 6,000 | 변수 223 | WT 평균 1.000 (모집단 가중)


## 1. 코어 2지표 — 신뢰·다양성 지수 (Task 02·03 SSOT)

In [2]:
# 신뢰·다양성·NCHI·규칙기반 페르소나 (모두 SSOT 함수)
T = F.trust_index(df)          # 신뢰 0~100 (코어 22문항 z평균)
D = F.diversity_index(df)      # 다양성 0~100 (Richness)
nchi = F.nchi(T, D)            # 기하평균 건강지수
persona_rule = F.persona_quadrant(T, D)   # 중앙값 4사분면(운영 기준)

dist = pd.DataFrame({"신뢰": T, "다양성": D, "NCHI": nchi}).describe().round(2)
print(dist.loc[["mean", "std", "min", "25%", "50%", "75%", "max"]])
print(f"\n신뢰 ↔ 다양성 상관 r = {T.corr(D):.3f}  → 두 축 독립(Task 03 재확인)")


          신뢰     다양성   NCHI
mean   62.05   26.08  38.02
std    12.16   15.69  13.70
min     1.00    1.00   4.16
25%    54.36    9.25  26.20
50%    63.18   25.75  39.47
75%    70.45   34.00  47.38
max   100.00  100.00  84.50



신뢰 ↔ 다양성 상관 r = 0.077  → 두 축 독립(Task 03 재확인)


## 2. 표준화 + K 스윕 (엘보우·실루엣)

표준화된 [신뢰, 다양성]에서 k=2~7 군집을 적합하고 inertia(엘보우)·실루엣을 비교한다.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

both = pd.DataFrame({"trust": T, "diversity": D}).dropna()
Xz = StandardScaler().fit_transform(both)

rows = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xz)
    rows.append({"k": k, "inertia": round(km.inertia_, 1),
                 "silhouette": round(silhouette_score(Xz, km.labels_), 3)})
sweep = pd.DataFrame(rows).set_index("k")
print(sweep)
print("\n→ 실루엣 0.32~0.36 평탄(k=5 미세 최고). 통계지표가 k를 강제하지 않음 →")
print("  해석가능성(4사분면 정합)으로 k=4 채택.")


   inertia  silhouette
k                     
2   7985.3       0.321
3   5195.7       0.357
4   4206.5       0.352
5   3406.7       0.364
6   2876.7       0.356
7   2478.4       0.356

→ 실루엣 0.32~0.36 평탄(k=5 미세 최고). 통계지표가 k를 강제하지 않음 →
  해석가능성(4사분면 정합)으로 k=4 채택.


### 2-b. 보조지표(검증·회피) 포함은 왜 배제했나

검증·회피를 군집 차원에 직접 넣은 4지표 입력과 비교 — 데이터가 배제를 지지함을 기록.

In [4]:
# 검증 프록시(z평균→1~100)·회피 플래그
vp = F.mask_special(df, F.VERIFY_PROXY)
verify = F._scale_1_100(pd.DataFrame(StandardScaler().fit_transform(vp),
                                     index=vp.index, columns=vp.columns).mean(axis=1))
avoid = F.avoid_flag(df)

ext = pd.DataFrame({"trust": T, "diversity": D, "verify": verify, "avoid": avoid}).dropna()
Xe = StandardScaler().fit_transform(ext)
sil_ext = {k: round(silhouette_score(Xe, KMeans(n_clusters=k, n_init=10, random_state=42)
                    .fit_predict(Xe)), 3) for k in range(2, 6)}
print(f"검증 결측율 {verify.isna().mean():.1%} | 회피 해당 {avoid.mean():.1%}(희소)")
print(f"4지표 표본 n={len(ext):,} (2지표 {len(both):,} 대비 {len(both)-len(ext):,} 손실)")
print("4지표 실루엣:", sil_ext, "→ 2지표(0.35대) 대비 0.25대로 하락")
print("결론: 검증·회피는 군집 입력에서 제외, 군집 후 프로파일 변수로만 활용.")


검증 결측율 30.1% | 회피 해당 4.3%(희소)
4지표 표본 n=4,194 (2지표 6,000 대비 1,806 손실)
4지표 실루엣: {2: 0.261, 3: 0.251, 4: 0.253, 5: 0.244} → 2지표(0.35대) 대비 0.25대로 하락
결론: 검증·회피는 군집 입력에서 제외, 군집 후 프로파일 변수로만 활용.


## 3. k=4 군집 × 4사분면 교차검증 (하이브리드 핵심)

K-means가 규칙기반 4사분면을 얼마나 재발견하는지 — 교차표·ARI·라벨 일치율.

In [5]:
labels, centers, km = F.kmeans_personas(T, D, k=4)
print("=== 군집 중심 (원척도) ===")
cen = centers.copy()
cen["대응_사분면"] = [F.PERSONA_LABELS[(r.trust >= T.median(), r.diversity >= D.median())]
                      for _, r in cen.iterrows()]
print(cen.round(1))

cmap = F.map_cluster_to_persona(centers, T.median(), D.median())
km_persona = labels.map(cmap)

print("\n=== K-means 군집 × 규칙 4사분면 교차표 ===")
print(pd.crosstab(labels, persona_rule))

ari = F.adjusted_rand(persona_rule, km_persona)
m = persona_rule.notna() & km_persona.notna()
agree = (persona_rule[m] == km_persona[m]).mean()
print(f"\nARI(조정 랜드지수) = {ari:.3f}  |  동일 라벨 비율 = {agree:.1%}")
print("→ 데이터 주도 군집이 4사분면을 독립 재발견(상당 일치). 중앙값 임계의 타당화.")


=== 군집 중심 (원척도) ===
   trust  diversity   대응_사분면
0   49.2       10.2    이중취약형
1   70.0       44.4  건강한 소비자
2   70.3       16.8    신뢰편향형
3   51.9       33.8  비판적 탐색형

=== K-means 군집 × 규칙 4사분면 교차표 ===
col_0    건강한 소비자  비판적 탐색형  신뢰편향형  이중취약형
cluster                                
0              0        5      0   1042
1           1154      209      0      0
2            737       86   1109    182
3              0     1476      0      0

ARI(조정 랜드지수) = 0.561  |  동일 라벨 비율 = 79.7%
→ 데이터 주도 군집이 4사분면을 독립 재발견(상당 일치). 중앙값 임계의 타당화.


## 4. 페르소나별 프로파일 (가중) — 웹 데모 입력 사양

운영 기준인 **규칙기반 4사분면**으로 인구(연령·성별·권역)·매체 이용을 프로파일링.

In [6]:
ageb = F.age_band(df); reg = F.region_band(df)
order = ["건강한 소비자", "비판적 탐색형", "신뢰편향형", "이중취약형"]

prof = []
for lab in order:
    mm = (persona_rule == lab)
    prof.append({
        "페르소나": lab,
        "가중%": round(w[mm].sum() / w.sum() * 100, 1),
        "n": int(mm.sum()),
        "평균연령": round(F.wmean(df[F.AGE][mm], w[mm]), 1),
        "여성%": round(w[mm & (df[F.GENDER] == 2)].sum() / w[mm].sum() * 100, 1),
        "신뢰": round(F.wmean(T[mm], w[mm]), 1),
        "다양성": round(F.wmean(D[mm], w[mm]), 1),
        "NCHI": round(F.wmean(nchi[mm], w[mm]), 1),
    })
prof = pd.DataFrame(prof).set_index("페르소나")
print("=== 페르소나 프로파일 (가중) ===")
print(prof)


=== 페르소나 프로파일 (가중) ===
          가중%     n  평균연령   여성%    신뢰   다양성  NCHI
페르소나                                             
건강한 소비자  34.8  1891  47.5  47.5  71.8  37.1  51.0
비판적 탐색형  28.9  1776  47.5  46.5  54.3  35.8  43.5
신뢰편향형    17.8  1109  57.3  59.5  71.3  10.9  27.0
이중취약형    18.5  1224  53.8  52.2  51.7  10.6  22.3


In [7]:
# 연령대 분포(가중, 행합 100%)
print("=== 페르소나 × 연령대 (가중%) ===")
age_ct = pd.crosstab(persona_rule, ageb, values=w, aggfunc="sum", normalize="index") * 100
print(age_ct.reindex(order).round(1))

print("\n=== 페르소나 × 권역 (가중%) ===")
reg_ct = pd.crosstab(persona_rule, reg, values=w, aggfunc="sum", normalize="index") * 100
print(reg_ct.reindex(order).round(1))


=== 페르소나 × 연령대 (가중%) ===
DQ3      19-29  30-39  40-49  50-59  60-69   70+
row_0                                           
건강한 소비자   15.2   18.8   20.6   21.8   16.2   7.4
비판적 탐색형   15.0   18.1   22.2   22.0   15.3   7.5
신뢰편향형     13.8    8.5    8.7   14.2   20.2  34.5
이중취약형     14.4   12.7   11.7   16.0   21.4  23.8

=== 페르소나 × 권역 (가중%) ===
SQ1       강원   수도권    영남   제주    충청    호남
row_0                                    
건강한 소비자  3.3  60.9  15.9  0.9   9.2   9.8
비판적 탐색형  2.6  51.7  24.7  1.7  10.9   8.4
신뢰편향형    3.1  44.8  27.7  1.2  14.5   8.7
이중취약형    2.7  37.7  35.6  1.4  11.0  11.6


In [8]:
# 매체 이용 프로파일: 페르소나별 평균 Richness(매체유형 폭)·주이용 매체유형
days = F.news_days_matrix(df)
div_m = F.diversity_metrics(df)
print("=== 페르소나별 다양성 구성(가중) ===")
comp = []
for lab in order:
    mm = (persona_rule == lab)
    comp.append({
        "페르소나": lab,
        "평균_매체수(Richness)": round(F.wmean(div_m["richness"][mm], w[mm]), 2),
        "유효매체수(expH)": round(F.wmean(div_m["effective_N"][mm], w[mm]), 2),
        "집중도HHI": round(F.wmean(div_m["hhi"][mm], w[mm]), 3),
    })
print(pd.DataFrame(comp).set_index("페르소나"))

# 페르소나별 매체유형 이용률(일수>0, 가중) — 상위 매체
print("\n=== 페르소나별 매체유형 이용률 (가중%, 주요 6종) ===")
use = (days > 0).astype(float)
top6 = ["TV뉴스", "포털뉴스", "인터넷뉴스", "종이신문", "SNS뉴스", "유튜브" if "유튜브" in use.columns else "동영상뉴스"]
top6 = [c for c in top6 if c in use.columns]
rows = []
for lab in order:
    mm = (persona_rule == lab)
    rows.append({"페르소나": lab, **{c: round(F.wmean(use[c][mm], w[mm]) * 100, 1) for c in top6}})
print(pd.DataFrame(rows).set_index("페르소나"))


=== 페르소나별 다양성 구성(가중) ===
         평균_매체수(Richness)  유효매체수(expH)  집중도HHI
페르소나                                          
건강한 소비자              4.38         4.13   0.275
비판적 탐색형              4.22         3.97   0.285
신뢰편향형                1.20         1.25   0.804
이중취약형                1.17         1.29   0.714

=== 페르소나별 매체유형 이용률 (가중%, 주요 6종) ===
         TV뉴스  포털뉴스  인터넷뉴스  종이신문  SNS뉴스  동영상뉴스
페르소나                                          
건강한 소비자  88.5  95.6   99.3   9.7   13.2   48.7
비판적 탐색형  83.7  95.9   99.9   8.9   11.7   44.4
신뢰편향형    76.8  14.8   19.7   5.9    0.1    0.2
이중취약형    66.2  15.5   23.1   7.3    0.0    1.0


## 5. 강건성표 — 설계 선택 민감도

군집 수 k·표준화·집계(기하/산술)·임계(중앙/평균)를 흔들어 페르소나 분포·NCHI 안정성을 점검.

In [9]:
# 5-a. 집계 방식(기하 vs 산술) × 임계(중앙 vs 평균) → 4사분면 분포 변동
def quad_dist(t_thr, d_thr):
    p = F.persona_quadrant(T, D, t_thr, d_thr)
    return {lab: round(w[p == lab].sum() / w.sum() * 100, 1) for lab in order}

rob = pd.DataFrame({
    "중앙값임계": quad_dist(T.median(), D.median()),
    "평균임계":   quad_dist(T.mean(),   D.mean()),
}).T
print("=== 임계 민감도: 4사분면 가중% ===")
print(rob)

print("\n=== 집계 방식 민감도: NCHI 분포 ===")
agg = pd.DataFrame({
    "기하평균": F.nchi(T, D, "geometric").describe()[["mean", "50%", "std"]],
    "산술평균": F.nchi(T, D, "arithmetic").describe()[["mean", "50%", "std"]],
}).round(2)
print(agg)


=== 임계 민감도: 4사분면 가중% ===
       건강한 소비자  비판적 탐색형  신뢰편향형  이중취약형
중앙값임계     34.8     28.9   17.8   18.5
평균임계      23.7     16.9   32.2   27.2

=== 집계 방식 민감도: NCHI 분포 ===
       기하평균   산술평균
mean  38.02  44.07
50%   39.47  43.78
std   13.70  10.29


In [10]:
# 5-b. k 민감도(3·4·5) + 표준화 방식(z vs minmax) → 실루엣
from sklearn.preprocessing import MinMaxScaler
Xmm = MinMaxScaler().fit_transform(both)
rob_k = []
for name, Xs in [("z표준화", Xz), ("minmax", Xmm)]:
    for k in (3, 4, 5):
        lab = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(Xs)
        rob_k.append({"표준화": name, "k": k, "실루엣": round(silhouette_score(Xs, lab), 3)})
print("=== k·표준화 민감도 ===")
print(pd.DataFrame(rob_k).pivot(index="k", columns="표준화", values="실루엣"))
print("\n→ k=4는 z·minmax 모두에서 안정. 페르소나 분포는 임계 선택에 다소 민감하나")
print("  순위·해석(건강>비판≈신뢰편향>이중취약 구조)은 불변.")


=== k·표준화 민감도 ===
표준화  minmax   z표준화
k                 
3     0.355  0.357
4     0.358  0.352
5     0.377  0.364

→ k=4는 z·minmax 모두에서 안정. 페르소나 분포는 임계 선택에 다소 민감하나
  순위·해석(건강>비판≈신뢰편향>이중취약 구조)은 불변.


## 6. 웹 데모 자가진단 — 입력 사양

자가진단은 **규칙기반 4사분면(운영)** 을 사용. 사용자 응답 → 신뢰·다양성 점수 → 사분면 배정 → 페르소나 카드.

In [11]:
spec = pd.DataFrame([
    {"축": "신뢰", "입력": "코어 신뢰 문항 축약(예: Q87·Q88·Q94 핵심 6~8문항, 5점)",
     "산출": "z평균 → 1~100", "임계": f"중앙값 {T.median():.1f}"},
    {"축": "다양성", "입력": "최근 1주 이용 뉴스 매체유형 체크(12종 중)",
     "산출": "Richness(체크 수) → 1~100", "임계": f"중앙값 {D.median():.1f}"},
], )
print("=== 자가진단 입력 사양 ===")
print(spec.to_string(index=False))

print("\n=== 결과 카드: 페르소나별 한줄 정의 + 처방 ===")
cards = {
    "건강한 소비자": "신뢰·다양성 모두 높음 → 현 습관 유지, 교차검증 루틴화",
    "비판적 탐색형": "다양하나 신뢰 낮음 → 신뢰할 원출처 1~2곳 고정 추천",
    "신뢰편향형":   "신뢰 높으나 편식 → 안 보던 매체유형 1종 추가 권장",
    "이중취약형":   "둘 다 낮음 → 가장 쉬운 1매체부터 신뢰·확장 동시 코칭",
}
for lab in order:
    pr = prof.loc[lab]
    print(f"- {lab} (인구 {pr['가중%']}%, 평균 {pr['평균연령']:.0f}세, NCHI {pr['NCHI']:.0f}): {cards[lab]}")

print("\n[근거 보존] K-means 검증: ARI", round(ari, 3), "/ 라벨 일치", f"{agree:.0%}",
      "→ 중앙값 4사분면이 데이터 군집과 정합.")


=== 자가진단 입력 사양 ===
  축                                       입력                     산출       임계
 신뢰 코어 신뢰 문항 축약(예: Q87·Q88·Q94 핵심 6~8문항, 5점)            z평균 → 1~100 중앙값 63.2
다양성               최근 1주 이용 뉴스 매체유형 체크(12종 중) Richness(체크 수) → 1~100 중앙값 25.8

=== 결과 카드: 페르소나별 한줄 정의 + 처방 ===
- 건강한 소비자 (인구 34.8%, 평균 48세, NCHI 51): 신뢰·다양성 모두 높음 → 현 습관 유지, 교차검증 루틴화
- 비판적 탐색형 (인구 28.9%, 평균 48세, NCHI 44): 다양하나 신뢰 낮음 → 신뢰할 원출처 1~2곳 고정 추천
- 신뢰편향형 (인구 17.8%, 평균 57세, NCHI 27): 신뢰 높으나 편식 → 안 보던 매체유형 1종 추가 권장
- 이중취약형 (인구 18.5%, 평균 54세, NCHI 22): 둘 다 낮음 → 가장 쉬운 1매체부터 신뢰·확장 동시 코칭

[근거 보존] K-means 검증: ARI 0.561 / 라벨 일치 80% → 중앙값 4사분면이 데이터 군집과 정합.
